In [ ]:
# STEP 1: Install all required libraries
!pip install -q kagglehub pandas
!pip install -q langchain langchain-openai langchain-huggingface langchain-chroma langchain-core langchain-community
!pip install -q chromadb sentence-transformers
!pip install -q gradio openai
!pip install -q pypdf easyocr pillow
!pip install -q langchain-text-splitters
!pip install -q langchain langchain-community langchain-core langchain-openai

print("✅ Step 1 completed: All libraries installed")


✅ Step 1 completed: All libraries installed
✅ Step 1 completed: All libraries installed


In [ ]:
# STEP 2: Download Uber Eats dataset
import kagglehub
import pandas as pd

# Download dataset
path = kagglehub.dataset_download("ahmedshahriarsakib/uber-eats-usa-restaurants-menus/versions/1")
print(f"Dataset path: {path}")

# Load CSV files
restaurants_path = path + "/restaurants.csv"
menus_path = path + "/restaurant-menus.csv"

restaurants_df = pd.read_csv(restaurants_path)
menus_df = pd.read_csv(menus_path)

print(f"✅ Loaded {len(restaurants_df)} restaurants")
print(f"✅ Loaded {len(menus_df)} menu items")
print(f"\nRestaurants columns: {restaurants_df.columns.tolist()}")
print(f"Menu columns: {menus_df.columns.tolist()}")

Using Colab cache for faster access to the 'uber-eats-usa-restaurants-menus' dataset.
Dataset path: /kaggle/input/uber-eats-usa-restaurants-menus
✅ Loaded 5000 restaurants
✅ Loaded 403240 menu items

Restaurants columns: ['id', 'position', 'name', 'score', 'ratings', 'category', 'price_range', 'full_address', 'zip_code', 'lat', 'lng']
Menu columns: ['restaurant_id', 'category', 'name', 'description', 'price']
Using Colab cache for faster access to the 'uber-eats-usa-restaurants-menus' dataset.
Dataset path: /kaggle/input/uber-eats-usa-restaurants-menus
✅ Loaded 5000 restaurants
✅ Loaded 403240 menu items

Restaurants columns: ['id', 'position', 'name', 'score', 'ratings', 'category', 'price_range', 'full_address', 'zip_code', 'lat', 'lng']
Menu columns: ['restaurant_id', 'category', 'name', 'description', 'price']


In [ ]:
# STEP 3: Merge restaurants and menus data
import pandas as pd

# Merge on restaurant_id
df = menus_df.merge(
    restaurants_df,
    left_on="restaurant_id",
    right_on="id",
    how="left"
)

# Drop rows with missing names
df = df.dropna(subset=["name_x", "name_y"])

print(f"✅ Merged dataset: {len(df)} items")
print(f"\nColumns after merge: {df.columns.tolist()}")
print(f"\nSample data:")
print(df[['name_y', 'name_x', 'price']].head())

✅ Merged dataset: 403240 items

Columns after merge: ['restaurant_id', 'category_x', 'name_x', 'description', 'price', 'id', 'position', 'name_y', 'score', 'ratings', 'category_y', 'price_range', 'full_address', 'zip_code', 'lat', 'lng']

Sample data:
                              name_y                               name_x  \
0  PJ Fresh (224 Daniel Payne Drive)              Extra Large Meat Lovers   
1  PJ Fresh (224 Daniel Payne Drive)                  Extra Large Supreme   
2  PJ Fresh (224 Daniel Payne Drive)                Extra Large Pepperoni   
3  PJ Fresh (224 Daniel Payne Drive)  Extra Large BBQ Chicken &amp; Bacon   
4  PJ Fresh (224 Daniel Payne Drive)                 Extra Large 5 Cheese   

       price  
0  15.99 USD  
1  15.99 USD  
2  14.99 USD  
3  15.99 USD  
4  14.99 USD  
✅ Merged dataset: 403240 items

Columns after merge: ['restaurant_id', 'category_x', 'name_x', 'description', 'price', 'id', 'position', 'name_y', 'score', 'ratings', 'category_y', 'price_range',

In [ ]:
# STEP 4: Create LangChain Documents from dataset + knowledge base docs

from langchain_core.documents import Document
import pandas as pd

# ── Part A: Convert CSV data into LangChain Documents ──────────────────────────
df_sample = df.sample(min(5000, len(df)), random_state=42)
print(f"Using {len(df_sample)} menu items from dataset...")

csv_documents = []
for _, row in df_sample.iterrows():
    rating    = row['score']       if pd.notna(row['score'])       else "No rating"
    desc      = row['description'] if pd.notna(row['description']) else "No description"
    price     = row['price']       if pd.notna(row['price'])       else "Price not listed"
    address   = row['full_address']if pd.notna(row['full_address'])else "Address not listed"
    category  = row['category_y'] if pd.notna(row['category_y'])  else "Uncategorized"

    text = (
        f"Restaurant: {row['name_y']}\n"
        f"Category: {category}\n"
        f"Location: {address}\n"
        f"Rating: {rating}\n"
        f"Item: {row['name_x']}\n"
        f"Description: {desc}\n"
        f"Price: {price}"
    )

    csv_documents.append(Document(
        page_content=text,
        metadata={
            "source": "uber_eats_dataset",
            "restaurant": str(row['name_y']),
            "category": str(category)
        }
    ))

print(f"✅ Created {len(csv_documents)} menu item documents")


# ── Part B: Your knowledge base docs (FAQ, policies, etc.) ─────────────────────
# 👇 EDIT THESE with your real content — or replace with file uploads if you prefer

kb_documents = [

    Document(
        page_content="""FAQ - Frequently Asked Questions

Q: Do you offer halal food?
A: Yes, all our chicken and lamb dishes are halal-certified. Look for the (H) symbol on the menu.

Q: Do you have vegetarian options?
A: Yes, we have a dedicated vegetarian section including falafel, veggie wraps, and salads.

Q: Do you have vegan options?
A: Yes, our falafel bowl, hummus plate, and garden salad are fully vegan.

Q: Can I customize my order?
A: Yes, most items can be customized. Just mention your preferences when ordering.

Q: Do you cater for large groups?
A: Yes, we offer catering packages for groups of 10 or more. Contact us for pricing.""",
        metadata={"source": "faq", "type": "knowledge_base"}
    ),

    Document(
        page_content="""Allergen Information

Contains Gluten: All wraps, burgers, pasta dishes, and breaded items.
Contains Dairy: Cheese pizza, cream-based pasta, yogurt dips.
Contains Nuts: Baklava, some salad dressings — always ask staff.
Contains Eggs: Caesar salad dressing, some sauces.
Gluten-Free Options: Rice bowls, grilled meats, salads (no croutons).
Nut-Free: Most savory dishes — but cross-contamination is possible.

If you have a severe allergy, please inform staff before ordering.""",
        metadata={"source": "allergen_sheet", "type": "knowledge_base"}
    ),

    Document(
        page_content="""Opening Hours and Branch Locations

Main Branch - Downtown:
  Monday to Thursday: 10:00 AM – 11:00 PM
  Friday & Saturday:  10:00 AM – 12:00 AM (midnight)
  Sunday:             12:00 PM – 10:00 PM

Second Branch - Mall Location:
  Daily: 10:00 AM – 10:00 PM (follows mall hours)

Public Holidays: Hours may vary — check our Instagram for updates.""",
        metadata={"source": "opening_hours", "type": "knowledge_base"}
    ),

    Document(
        page_content="""Delivery and Pickup Policy

Delivery:
- Available via our website, app, or third-party apps (Uber Eats, Talabat).
- Delivery radius: up to 10 km from each branch.
- Minimum order for delivery: $15.
- Average delivery time: 30–45 minutes.
- Delivery fee: $2–$5 depending on distance.

Pickup:
- Available at all branches.
- Order ahead via app or phone to reduce wait time.
- Ready in 15–20 minutes for most orders.""",
        metadata={"source": "delivery_policy", "type": "knowledge_base"}
    ),

    Document(
        page_content="""Reservation Policy

- Reservations accepted for groups of 4 or more.
- Book via phone, website, or by messaging us on WhatsApp.
- We hold reservations for 15 minutes past the booked time.
- For large groups (10+), 48-hour advance booking is required.
- Cancellations must be made at least 2 hours before the reservation.
- Special occasion setups (birthday cakes, decorations) available on request.""",
        metadata={"source": "reservation_policy", "type": "knowledge_base"}
    ),

    Document(
        page_content="""Catering Menu and Services

We offer full catering for events, corporate lunches, and private gatherings.

Package A (Minimum 20 people) — $18/person:
  - Choice of 2 mains, 2 sides, bread, water.

Package B (Minimum 30 people) — $25/person:
  - Choice of 3 mains, 3 sides, salads, dessert, soft drinks.

Custom packages available on request.
All catering includes setup and serving staff.
Contact: catering@restaurant.com | WhatsApp: +1-800-FOOD""",
        metadata={"source": "catering_menu", "type": "knowledge_base"}
    ),

]

print(f"✅ Created {len(kb_documents)} knowledge base documents")


# ── Part C: Combine both ───────────────────────────────────────────────────────
documents = csv_documents + kb_documents
print(f"\n📚 Total documents ready for RAG: {len(documents)}")
print("   ├── Menu items (CSV):    ", len(csv_documents))
print("   └── Knowledge base docs: ", len(kb_documents))
print("\n✅ Step 4 complete — ready for Step 5 (vector DB)")

Using 5000 menu items from dataset...
✅ Created 5000 menu item documents
✅ Created 6 knowledge base documents

📚 Total documents ready for RAG: 5006
   ├── Menu items (CSV):     5000
   └── Knowledge base docs:  6

✅ Step 4 complete — ready for Step 5 (vector DB)
Using 5000 menu items from dataset...
✅ Created 5000 menu item documents
✅ Created 6 knowledge base documents

📚 Total documents ready for RAG: 5006
   ├── Menu items (CSV):     5000
   └── Knowledge base docs:  6

✅ Step 4 complete — ready for Step 5 (vector DB)


In [ ]:
# STEP 5: Split documents and build vector database
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(documents)
print(f"✅ Split into {len(split_docs)} chunks")

# Load embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("✅ Embedding model loaded")

# Build vector database
print("Building vector database (may take 1-2 minutes)...")
db = Chroma.from_documents(split_docs, embeddings)
print(f"✅ Vector DB ready — {db._collection.count()} vectors stored")

✅ Split into 6132 chunks
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded
Building vector database (may take 1-2 minutes)...
✅ Vector DB ready — 6132 vectors stored
✅ Split into 6132 chunks
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded
Building vector database (may take 1-2 minutes)...
✅ Vector DB ready — 12264 vectors stored


In [ ]:
# STEP 6: Setup OpenAI
from openai import OpenAI
from langchain_openai import ChatOpenAI

api_key = .....
# OpenAI client (used for function calling)
client = OpenAI(api_key=api_key)

# LangChain LLM (used for RAG chain)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    openai_api_key=api_key
)

# Test connection
try:
    test = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=10,
        messages=[{"role": "user", "content": "Say OK"}]
    )
    print("✅ OpenAI connection successful:", test.choices[0].message.content)
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ OpenAI connection successful: OK!
✅ OpenAI connection successful: OK!


In [ ]:
!pip install -q langchain-core langchain-community langgraph langchain-openai

In [ ]:
# STEP 7: Build RAG chain with conversation memory + Arabic support

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

# --- Prompt ---
system_template = """You are a friendly and knowledgeable restaurant customer support assistant.
Your job is to help customers with questions about menu items, prices, ingredients, allergens,
halal options, opening hours, reservations, delivery, and catering.

Rules:
1. Answer ONLY using the context provided below.
2. If the answer is not in the context, say: "I don't have that information, please contact the restaurant directly."
3. NEVER guess or make up information.
4. Detect the user's language and always respond in the SAME language.
   - If the user writes in Arabic, respond fully in Arabic.
   - If the user writes in English, respond in English.
5. Be concise, warm, and helpful.

Few-shot examples:
User: Do you have halal options?
A: Yes! We offer several halal-certified dishes. Our chicken shawarma, grilled lamb, and falafel wrap are all halal.

User: هل يوجد خيارات نباتية؟
A: نعم! لدينا عدة خيارات نباتية منها: سلطة الفتوش، شاورما الخضار، والفلافل.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# --- Manual memory store ---
chat_history = []

# --- Retriever ---
retriever = db.as_retriever(search_type="mmr",search_kwargs={"k": 3})

# --- Helper to format retrieved docs ---
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# --- Chain ---
rag_chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm
    | StrOutputParser()
)

# --- Ask function with memory ---
def ask(question: str) -> str:
    global chat_history
    response = rag_chain.invoke({
        "question": question,
        "chat_history": chat_history
    })
    # Append to memory
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    return response

print("✅ RAG chain with memory ready")

# --- Quick test ---
test_questions = [
    "Do you have vegetarian options?",
    "What was my first question?",    # tests memory
    "هل يوجد خيارات حلال؟"           # Arabic test
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")

✅ RAG chain with memory ready
Q: Do you have vegetarian options?
A: Yes, we have a dedicated vegetarian section including falafel, veggie wraps, and salads.

Q: What was my first question?
A: I don't have that information, please contact the restaurant directly.

Q: هل يوجد خيارات حلال؟
A: نعم! لدينا خيارات حلال، بما في ذلك أطباق مثل دجاج الشاورما ولحم الضأن المشوي.

✅ RAG chain with memory ready
Q: Do you have vegetarian options?
A: Yes, we have a dedicated vegetarian section including falafel, veggie wraps, and salads.

Q: What was my first question?
A: I don't have that information, please contact the restaurant directly.

Q: هل يوجد خيارات حلال؟
A: نعم! نقدم عدة أطباق معتمدة حلال. تشمل خياراتنا شاورما الدجاج، لحم الضأن المشوي، ولفائف الفلافل.



In [ ]:
# STEP 8: Function Calling for Structured Outputs

import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.prompts import PromptTemplate

# ──────────────────────────────────────────────
# 1. Define structured output schemas (Pydantic)
# ──────────────────────────────────────────────

class OrderSummary(BaseModel):
    """Extracted order details from user message."""
    items: list[str] = Field(description="List of ordered items")
    quantities: list[int] = Field(description="Quantity for each item")
    special_requests: Optional[str] = Field(description="Any special requests or notes")
    delivery_or_pickup: Optional[str] = Field(description="'delivery' or 'pickup'")

class ReservationRequest(BaseModel):
    """Extracted reservation details from user message."""
    name: Optional[str] = Field(description="Customer name")
    date: Optional[str] = Field(description="Reservation date")
    time: Optional[str] = Field(description="Reservation time")
    party_size: Optional[int] = Field(description="Number of guests")
    special_requests: Optional[str] = Field(description="Any special requests")

class ComplaintForm(BaseModel):
    """Extracted complaint details from user message."""
    issue_type: Optional[str] = Field(description="Type of issue: food quality, delivery, service, billing, other")
    description: str = Field(description="Full description of the complaint")
    order_number: Optional[str] = Field(description="Order number if mentioned")
    preferred_resolution: Optional[str] = Field(description="What the customer wants: refund, replacement, apology, other")

class CateringInquiry(BaseModel):
    """Extracted catering inquiry details from user message."""
    event_type: Optional[str] = Field(description="Type of event: wedding, corporate, birthday, other")
    guest_count: Optional[int] = Field(description="Expected number of guests")
    date: Optional[str] = Field(description="Event date")
    budget: Optional[str] = Field(description="Budget range if mentioned")
    dietary_requirements: Optional[str] = Field(description="Any dietary requirements")

# ──────────────────────────────────────────────
# 2. Intent detector
# ──────────────────────────────────────────────

intent_prompt = PromptTemplate.from_template("""
Classify the user message into one of these intents:
- order
- reservation
- complaint
- catering
- general

Respond with ONLY one word (the intent). No explanation.

User message: {message}
""")

intent_chain = intent_prompt | llm | StrOutputParser()

def detect_intent(message: str) -> str:
    intent = intent_chain.invoke({"message": message}).strip().lower()
    # Fallback to general if unexpected output
    valid = ["order", "reservation", "complaint", "catering", "general"]
    return intent if intent in valid else "general"

# ──────────────────────────────────────────────
# 3. Structured extractors (with_structured_output)
# ──────────────────────────────────────────────

def extract_order(message: str) -> dict:
    structured_llm = llm.with_structured_output(OrderSummary)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract the order details from the user message. If a field is missing, leave it as None."),
        ("human", "{message}")
    ])
    chain = prompt | structured_llm
    result = chain.invoke({"message": message})
    return result.dict()

def extract_reservation(message: str) -> dict:
    structured_llm = llm.with_structured_output(ReservationRequest)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract the reservation details from the user message. If a field is missing, leave it as None."),
        ("human", "{message}")
    ])
    chain = prompt | structured_llm
    result = chain.invoke({"message": message})
    return result.dict()

def extract_complaint(message: str) -> dict:
    structured_llm = llm.with_structured_output(ComplaintForm)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract the complaint details from the user message. If a field is missing, leave it as None."),
        ("human", "{message}")
    ])
    chain = prompt | structured_llm
    result = chain.invoke({"message": message})
    return result.dict()

def extract_catering(message: str) -> dict:
    structured_llm = llm.with_structured_output(CateringInquiry)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract the catering inquiry details from the user message. If a field is missing, leave it as None."),
        ("human", "{message}")
    ])
    chain = prompt | structured_llm
    result = chain.invoke({"message": message})
    return result.dict()

# ──────────────────────────────────────────────
# 4. Master router — detects intent + extracts
# ──────────────────────────────────────────────

def handle_structured_request(message: str) -> dict:
    intent = detect_intent(message)
    print(f"🔍 Detected intent: {intent}")

    if intent == "order":
        data = extract_order(message)
    elif intent == "reservation":
        data = extract_reservation(message)
    elif intent == "complaint":
        data = extract_complaint(message)
    elif intent == "catering":
        data = extract_catering(message)
    else:
        # Fall back to normal RAG chain for general questions
        data = {"response": ask(message)}

    return {"intent": intent, "extracted": data}

print("✅ Function calling / structured output ready")

# ──────────────────────────────────────────────
# 5. Quick tests
# ──────────────────────────────────────────────

test_messages = [
    "I'd like to order 2 shawarmas and 1 falafel wrap, please deliver to my address",
    "I want to book a table for 4 people on Friday at 7pm, name is Sara",
    "My order was cold and arrived 1 hour late, I want a refund. Order #1042",
    "We're planning a corporate event for 80 people in December, what catering options do you have?",
    "Do you have gluten-free options?",   # general → RAG
]

for msg in test_messages:
    print(f"\n📩 Message: {msg}")
    result = handle_structured_request(msg)
    print(f"📦 Result:\n{json.dumps(result, indent=2, ensure_ascii=False)}")

✅ Function calling / structured output ready

📩 Message: I'd like to order 2 shawarmas and 1 falafel wrap, please deliver to my address
🔍 Detected intent: order


/tmp/ipykernel_25980/1067080189.py:81: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict()


📦 Result:
{
  "intent": "order",
  "extracted": {
    "items": [
      "shawarma",
      "falafel wrap"
    ],
    "quantities": [
      2,
      1
    ],
    "special_requests": null,
    "delivery_or_pickup": "delivery"
  }
}

📩 Message: I want to book a table for 4 people on Friday at 7pm, name is Sara
🔍 Detected intent: reservation


/tmp/ipykernel_25980/1067080189.py:91: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict()


📦 Result:
{
  "intent": "reservation",
  "extracted": {
    "name": "Sara",
    "date": "Friday",
    "time": "7pm",
    "party_size": 4,
    "special_requests": null
  }
}

📩 Message: My order was cold and arrived 1 hour late, I want a refund. Order #1042
🔍 Detected intent: complaint


...

📦 Result:
{
  "intent": "reservation",
  "extracted": {
    "name": "Sara",
    "date": "Friday",
    "time": "7pm",
    "party_size": 4,
    "special_requests": null
  }
}

📩 Message: My order was cold and arrived 1 hour late, I want a refund. Order #1042
🔍 Detected intent: complaint


/tmp/ipykernel_25980/1067080189.py:101: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict()


📦 Result:
{
  "intent": "complaint",
  "extracted": {
    "issue_type": "delivery",
    "description": "My order was cold and arrived 1 hour late.",
    "order_number": "1042",
    "preferred_resolution": "refund"
  }
}

📩 Message: We're planning a corporate event for 80 people in December, what catering options do you have?
🔍 Detected intent: catering


/tmp/ipykernel_25980/1067080189.py:111: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict()


📦 Result:
{
  "intent": "catering",
  "extracted": {
    "event_type": "corporate",
    "guest_count": 80,
    "date": "December",
    "budget": null,
    "dietary_requirements": null
  }
}

📩 Message: Do you have gluten-free options?
🔍 Detected intent: general
📦 Result:
{
  "intent": "general",
  "extracted": {
    "response": "Yes, we offer gluten-free options, including our corn tortillas."
  }
}


In [ ]:
# STEP 9: Image Input with OCR

# ──────────────────────────────────────────────
# 1. Install dependencies
# ──────────────────────────────────────────────
!pip install -q easyocr pillow

# ──────────────────────────────────────────────
# 2. Imports
# ──────────────────────────────────────────────
import easyocr
import numpy as np
from PIL import Image
import io

# Initialize EasyOCR (English + Arabic)
print("Loading OCR model... (first time takes ~1 min)")
ocr_reader = easyocr.Reader(['en', 'ar'], gpu=False)
print("✅ OCR ready")

# ──────────────────────────────────────────────
# 3. OCR extractor
# ──────────────────────────────────────────────

def extract_text_from_image(image) -> str:
    """
    Accepts: PIL Image, file path (str), or bytes
    Returns: extracted text as a single string
    """
    # Normalize input to numpy array
    if isinstance(image, str):
        img = Image.open(image).convert("RGB")
    elif isinstance(image, bytes):
        img = Image.open(io.BytesIO(image)).convert("RGB")
    elif isinstance(image, Image.Image):
        img = image.convert("RGB")
    else:
        img = image  # assume numpy array

    img_array = np.array(img)
    results = ocr_reader.readtext(img_array, detail=0, paragraph=True)
    extracted = "\n".join(results).strip()
    return extracted if extracted else "No text could be extracted from this image."

# ──────────────────────────────────────────────
# 4. OCR → RAG pipeline
# ──────────────────────────────────────────────

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ocr_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful restaurant assistant.
The user has uploaded an image (menu, receipt, flyer, or screenshot).
The text extracted from the image is provided below.

Your job:
1. Understand what the image contains.
2. Answer the user's question using BOTH the extracted image text AND the restaurant knowledge base context.
3. If the user writes in Arabic, respond in Arabic. Otherwise respond in English.
4. If you cannot find the answer, say: "I don't have that information, please contact the restaurant directly."

Extracted image text:
{ocr_text}

Knowledge base context:
{context}
"""),
    ("human", "{question}"),
])

def ask_with_image(image, question: str) -> str:
    """
    Full pipeline: image → OCR → RAG → answer
    """
    # Step 1: Extract text from image
    ocr_text = extract_text_from_image(image)
    print(f"\n📄 Extracted OCR text:\n{ocr_text}\n")

    # Step 2: Retrieve relevant docs from knowledge base
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Step 3: Generate answer
    chain = ocr_prompt | llm | StrOutputParser()
    response = chain.invoke({
        "ocr_text": ocr_text,
        "question": question,
        "context": context,
    })

    # Step 4: Save to memory
    from langchain_core.messages import HumanMessage, AIMessage
    chat_history.append(HumanMessage(content=f"[Image uploaded] {question}"))
    chat_history.append(AIMessage(content=response))

    return response

print("✅ OCR + RAG pipeline ready")

# ──────────────────────────────────────────────
# 5. Quick test (create a dummy menu image)
# ──────────────────────────────────────────────

from PIL import ImageDraw, ImageFont

def create_test_menu_image() -> Image.Image:
    """Creates a simple menu image for testing."""
    img = Image.new("RGB", (600, 400), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    menu_text = """RESTAURANT MENU

Chicken Shawarma     $8.99   (Halal)
Falafel Wrap         $6.99   (Vegan)
Grilled Salmon       $14.99
Fattoush Salad       $5.99   (Vegetarian)
Lemonade             $2.99

* All chicken dishes are halal certified
* Gluten-free options available on request"""
    draw.multiline_text((30, 30), menu_text, fill=(0, 0, 0), spacing=8)
    return img

# Generate and test
test_image = create_test_menu_image()
test_image.save("/tmp/test_menu.png")
print("🖼️ Test menu image created\n")

test_questions_ocr = [
    "What are the vegan options on this menu?",
    "How much does the shawarma cost?",
    "ما هي الأطباق الحلال في هذه القائمة؟"   # Arabic
]

for q in test_questions_ocr:
    print(f"Q: {q}")
    answer = ask_with_image(test_image, q)
    print(f"A: {answer}\n")
    print("-" * 50)

Loading OCR model... (first time takes ~1 min)
✅ OCR ready
✅ OCR + RAG pipeline ready
🖼️ Test menu image created

Q: What are the vegan options on this menu?

📄 Extracted OCR text:
١١٤١٧ T٥٧٥٥١٢؟٩٤
| Hنادا د $٥،99 Chicken ShawJFmIa ( vegani $3.99 Falalel Wrap $٦ 49٥ Grilled 5alm٥٨ $599 ( vegetariani Fattoush Silad $2.99 Lem٥٨٥٥e
Allchicken Uishesare halal certifled Gluteriree otlonis avallable ٥٨ equest

A: The vegan option on the menu is the Falafel Wrap, which is priced at $3.99. Additionally, the restaurant offers other vegan options such as the falafel bowl and hummus plate, although they are not listed in the extracted text.

--------------------------------------------------
Q: How much does the shawarma cost?

📄 Extracted OCR text:
١١٤١٧ T٥٧٥٥١٢؟٩٤
| Hنادا د $٥،99 Chicken ShawJFmIa ( vegani $3.99 Falalel Wrap $٦ 49٥ Grilled 5alm٥٨ $599 ( vegetariani Fattoush Silad $2.99 Lem٥٨٥٥e
Allchicken Uishesare halal certifled Gluteriree otlonis avallable ٥٨ equest

A: The Chicken Shawarma 

Loading OCR model... (first time takes ~1 min)
✅ OCR ready
✅ OCR + RAG pipeline ready
🖼️ Test menu image created

Q: What are the vegan options on this menu?

📄 Extracted OCR text:
١١٤١٧ T٥٧٥٥١٢؟٩٤
| Hنادا د $٥،99 Chicken ShawJFmIa ( vegani $3.99 Falalel Wrap $٦ 49٥ Grilled 5alm٥٨ $599 ( vegetariani Fattoush Silad $2.99 Lem٥٨٥٥e
Allchicken Uishesare halal certifled Gluteriree otlonis avallable ٥٨ equest

A: The menu includes the following vegan options:

1. Falafel Wrap - $3.99
2. Fattoush Salad - $2.99

Additionally, the restaurant offers a fully vegan falafel bowl, hummus plate, and garden salad, although these specific items are not listed on the menu you provided. If you need more details about the vegan options, please contact the restaurant directly.

--------------------------------------------------
Q: How much does the shawarma cost?

📄 Extracted OCR text:
١١٤١٧ T٥٧٥٥١٢؟٩٤
| Hنادا د $٥،99 Chicken ShawJFmIa ( vegani $3.99 Falalel Wrap $٦ 49٥ Grilled 5alm٥٨ $599 ( vegetariani Fa

In [ ]:
# STEP 10 (FIXED): Gradio UI with working image submission

import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage
import json

# ──────────────────────────────────────────────
# 1. Core response handler
# ──────────────────────────────────────────────

def process_message(user_message: str, image, history: list, stored_image) -> tuple:
    """
    Routes the message to the right pipeline.
    Uses stored_image (gr.State) so image persists when Send is clicked.
    """
    if not user_message.strip():
        return history, "", stored_image, stored_image, ""

    # Use whichever image is available (just uploaded OR previously stored)
    active_image = image if image is not None else stored_image

    structured_intents = ["order", "reservation", "complaint", "catering"]

    # ── Path 1: Image uploaded ──
    if active_image is not None:
        response = ask_with_image(active_image, user_message)
        source = "🖼️ OCR + RAG"

    # ── Path 2: Structured intent ──
    elif detect_intent(user_message) in structured_intents:
        result = handle_structured_request(user_message)
        intent = result["intent"]
        extracted = result["extracted"]

        if intent == "order":
            items_list = ", ".join(
                f"{q}x {i}" for i, q in zip(extracted.get("items", []), extracted.get("quantities", []))
            )
            response = f"✅ Order received!\n**Items:** {items_list}\n**Type:** {extracted.get('delivery_or_pickup', 'N/A')}\n**Special requests:** {extracted.get('special_requests', 'None')}"
        elif intent == "reservation":
            response = f"✅ Reservation noted!\n**Name:** {extracted.get('name', 'N/A')}\n**Date:** {extracted.get('date', 'N/A')}\n**Time:** {extracted.get('time', 'N/A')}\n**Guests:** {extracted.get('party_size', 'N/A')}"
        elif intent == "complaint":
            response = f"😔 We're sorry to hear that.\n**Issue:** {extracted.get('issue_type', 'N/A')}\n**Details:** {extracted.get('description', 'N/A')}\n**Resolution requested:** {extracted.get('preferred_resolution', 'N/A')}"
        elif intent == "catering":
            response = f"🎉 Catering inquiry received!\n**Event:** {extracted.get('event_type', 'N/A')}\n**Guests:** {extracted.get('guest_count', 'N/A')}\n**Date:** {extracted.get('date', 'N/A')}\n**Dietary needs:** {extracted.get('dietary_requirements', 'N/A')}"

        source = f"⚙️ Function Calling → `{intent}`"

    # ── Path 3: General RAG ──
    else:
        response = ask(user_message)
        source = "📚 RAG Knowledge Base"

    history.append((user_message, response))

    # Keep the image stored in state so follow-up questions still work
    # Set image component back to active_image so user can see it's still loaded
    return history, "", active_image, active_image, source


def store_image(image):
    """Called when user uploads an image — saves it to gr.State immediately."""
    return image


def reset_chat():
    global chat_history
    chat_history = []
    return [], "", None, None, ""


# ──────────────────────────────────────────────
# 2. Gradio UI
# ──────────────────────────────────────────────

with gr.Blocks(theme=gr.themes.Soft(), title="🍽️ Restaurant Assistant") as demo:

    gr.Markdown("""
    # 🍽️ Restaurant Customer Support Assistant
    **Ask me anything** about our menu, prices, allergens, halal options, reservations, delivery, and more.
    You can write in **English or Arabic** 🇬🇧🇸🇦, or **upload a menu/receipt image** 🖼️.
    > 💡 **How to use image:** Upload image first → wait for it to appear → then type your question → click Send
    """)

    # gr.State holds the image between interactions
    stored_image = gr.State(value=None)

    with gr.Row():

        # ── Left: Chat panel ──
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=480,
                bubble_full_width=False,
                show_copy_button=True,
            )
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Ask a question in English or Arabic...",
                    label="Your message",
                    scale=5,
                    lines=1,
                )
                send_btn = gr.Button("Send ➤", variant="primary", scale=1)

            clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

        # ── Right: Image + info panel ──
        with gr.Column(scale=1):
            image_input = gr.Image(
                label="📷 Upload Menu / Receipt / Flyer (upload BEFORE sending)",
                type="pil",
                height=250,
            )
            # Status indicator so user knows image is loaded
            image_status = gr.Textbox(
                label="📎 Image status",
                value="No image loaded",
                interactive=False,
                lines=1,
            )
            source_box = gr.Textbox(
                label="💡 Response source",
                interactive=False,
                lines=1,
            )
            gr.Markdown("""
            ### 💬 Try asking:
            - *Do you have halal options?*
            - *I want to order 2 shawarmas*
            - *Book a table for 4 on Friday at 7pm*
            - *My order arrived cold, I want a refund*
            - *هل يوجد خيارات نباتية؟*
            - *Upload a menu image and ask about it*
            """)

    gr.Examples(
        examples=[
            ["Do you have vegetarian options?", None],
            ["I'd like to order 2 shawarmas and 1 falafel, please deliver", None],
            ["Book a table for 4 people on Friday at 7pm, name is Sara", None],
            ["My order was cold and late, I want a refund. Order #1042", None],
            ["We need catering for a corporate event, 80 guests in December", None],
            ["هل يوجد خيارات حلال؟", None],
            ["ما هي أوقات العمل؟", None],
        ],
        inputs=[msg_box, image_input],
        label="📋 Example questions"
    )

    # ──────────────────────────────────────────────
    # 3. Event wiring
    # ──────────────────────────────────────────────

    # When image is uploaded → immediately store it in State + update status
    image_input.change(
        fn=store_image,
        inputs=[image_input],
        outputs=[stored_image],
    ).then(
        fn=lambda img: "✅ Image loaded — now type your question and click Send" if img is not None else "No image loaded",
        inputs=[image_input],
        outputs=[image_status],
    )

    # Send button
    send_btn.click(
        fn=process_message,
        inputs=[msg_box, image_input, chatbot, stored_image],
        outputs=[chatbot, msg_box, image_input, stored_image, source_box],
    )

    # Enter key
    msg_box.submit(
        fn=process_message,
        inputs=[msg_box, image_input, chatbot, stored_image],
        outputs=[chatbot, msg_box, image_input, stored_image, source_box],
    )

    # Clear
    clear_btn.click(
        fn=reset_chat,
        outputs=[chatbot, msg_box, image_input, stored_image, source_box],
    ).then(
        fn=lambda: "No image loaded",
        outputs=[image_status],
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_25980/1333081709.py:76: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="🍽️ Restaurant Assistant") as demo:
/tmp/ipykernel_25980/1333081709.py:92: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_25980/1333081709.py:92: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_25980/1333081709.py:92: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any 

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c3d3effd628c539fe4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔍 Detected intent: order


/tmp/ipykernel_25980/1067080189.py:81: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict()


🔍 Detected intent: reservation


/tmp/ipykernel_25980/1067080189.py:91: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict()


🔍 Detected intent: reservation


/tmp/ipykernel_25980/1067080189.py:91: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict()
